# CAB320 Assignment 2 - Transfer Learning
Anthony Vanderkop, Thierry Peynot, Frederic Maire (Jupyter Notebook template: 2025)


## Instructions:
The functions and classes defined in this module will be called by the marker without modification. 
You should complete the functions and classes according to their specified interfaces.

No partial marks will be awarded for functions that do not meet the specifications of the interfaces.


In [1]:
### LIBRARY IMPORTS ###
import os
import numpy as np
import tensorflow as tf
import keras.applications as ka
import keras

## Task 1
Implement the my_team()function 

In [2]:
def my_team():
    '''
    Return the list of the team members of this assignment submission as a list
    of triplet of the form (student_number, first_name, last_name)
    
    '''
    return [
        (11301406, 'Joshua', 'Rapsey'),
        (11290803, 'Joshua', 'Hentscher'),
        (11631996, 'Cooper', 'Smith')
    ]

In [3]:
my_team()

[(11301406, 'Joshua', 'Rapsey'),
 (11290803, 'Joshua', 'Hentscher'),
 (11631996, 'Cooper', 'Smith')]

## Task 2
Download the small_flower_dataset from Canvas and load the data

In [ ]:
def load_data(path='small_flower_dataset'):
    '''
    Load in the dataset from its home path. Path should be a string of the path
    to the home directory the dataset is found in. Returns the images as a
    numpy array and the associated class labels as a numpy array.
    '''
    class_names = [
    name for name in sorted(os.listdir(path))
    if os.path.isdir(os.path.join(path, name)) and not name.startswith('.')
    ]
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp')
    images = []
    labels = []

    for class_index, class_name in enumerate(class_names):
        class_path = os.path.join(path, class_name)
        for file_name in sorted(os.listdir(class_path)):
            file_path = os.path.join(class_path, file_name)
            if os.path.isfile(file_path) and file_name.lower().endswith(image_extensions):
                image_bytes = tf.io.read_file(file_path)
                image = tf.image.decode_image(image_bytes, channels=3, expand_animations=False)
                image = tf.image.resize(image, (224, 224))
                images.append(image.numpy())
                labels.append(class_index)

    images = ka.mobilenet_v2.preprocess_input(np.asarray(images, dtype=np.float32))
    labels = np.asarray(labels, dtype=np.int64)
    return images, labels

In [ ]:
dataset = load_data()

## Task 3
Prepare your training, validation and test sets for the non-accelerated version of transfer learning.

In [ ]:
def split_data(X, Y, train_fraction, randomize=False, eval_set=True):
    """
    Split the data into training and testing sets. If eval_set is True, also create
    an evaluation dataset. There should be two outputs if eval_set there should
    be three outputs (train, test, eval), otherwise two outputs (train, test).
    
    To see what type train, test, and eval should be, refer to the inputs of 
    transfer_learning().
    
    The split is performed in a stratified way so each class keeps a similar
    proportion in the train, evaluation, and test sets.
    """
    X = np.asarray(X)
    Y = np.asarray(Y)
    rng = np.random.default_rng()

    train_indices = []
    eval_indices = []
    test_indices = []

    for class_value in np.unique(Y):
        class_indices = np.where(Y == class_value)[0]
        if randomize:
            class_indices = class_indices.copy()
            rng.shuffle(class_indices)

        class_size = len(class_indices)
        remaining_required = 2 if eval_set and class_size >= 3 else 1 if (not eval_set and class_size >= 2) else 0
        train_count = int(round(train_fraction * class_size))
        train_count = max(0, min(train_count, class_size - remaining_required))

        remaining_indices = class_indices[train_count:]
        if eval_set:
            eval_count = len(remaining_indices) // 2
            test_count = len(remaining_indices) - eval_count
            train_indices.extend(class_indices[:train_count])
            eval_indices.extend(remaining_indices[:eval_count])
            test_indices.extend(remaining_indices[eval_count:eval_count + test_count])
        else:
            train_indices.extend(class_indices[:train_count])
            test_indices.extend(remaining_indices)

    if randomize:
        rng.shuffle(train_indices)
        rng.shuffle(eval_indices)
        rng.shuffle(test_indices)

    train_set = (X[train_indices], Y[train_indices])
    test_set = (X[test_indices], Y[test_indices])

    if eval_set:
        eval_set = (X[eval_indices], Y[eval_indices])
        return train_set, eval_set, test_set

    return train_set, test_set

In [ ]:
train_set, eval_set, test_set = split_data(dataset[0], dataset[1], 0.6, randomize=True, eval_set=True)

Report: Include details of how you have split the data to perform this training. Ensure the split is reasonable and does not introduce class imbalance during training

The data was split using a stratified 60/20/20 train/evaluation/test split.
Images were shuffled within each class before splitting so the five flower
classes stay balanced across all partitions.

## Task 4
Using the tf.keras.applications module download a pretrained MobileNetV2 network. 

In [ ]:
def load_model():
    '''
    Load in a model using the tf.keras.applications model and return it.
    The returned model is a pretrained MobileNetV2 network with the ImageNet
    weights loaded.
    '''
    return ka.MobileNetV2(weights='imagenet', include_top=True, input_shape=(224, 224, 3))
    

In [ ]:
model = load_model()

## Task 5
Replace the last layer of the downloaded neural network with a Dense layer of the appropriate shape for the 5 classes of the small flower dataset {(x1,t1), (x2,t2),..., (xm,tm)}.

## Task 6
Compile and train your model with an SGD optimizer using the following parameters learning_rate=0.01, momentum=0.0, nesterov=False. (NB: The SGD class description can be found at https://keras.io/api/optimizers/sgd/  )

In [ ]:
def transfer_learning(train_set, eval_set, model, parameters):
    '''
    Implement and perform standard transfer learning here.

    Inputs:
        - train_set: list or tuple of the training images and labels in the
            form (images, labels) for training the classifier
        - eval_set: list or tuple of the images and labels used in evaluating
            the model during training, in the form (images, labels)
        - model: an instance of tf.keras.applications.MobileNetV2
        - parameters: list or tuple of parameters to use during training:
            (learning_rate, momentum, nesterov)


    Outputs:
        - model : an instance of tf.keras.applications.MobileNetV2

    '''
    learning_rate, momentum, nesterov = parameters
    train_images, train_labels = train_set
    eval_images, eval_labels = eval_set

    for layer in model.layers:
        layer.trainable = False

    classifier = keras.layers.Dense(5, activation='softmax', name='flower_predictions')(model.layers[-2].output)
    model = keras.Model(inputs=model.input, outputs=classifier)
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=learning_rate,
            momentum=momentum,
            nesterov=nesterov,
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    history = model.fit(
        train_images,
        train_labels,
        validation_data=(eval_images, eval_labels),
        epochs=10,
        batch_size=32,
        verbose=1,
    )
    return model, history

In [ ]:
import os
print(sorted(os.listdir('small_flower_dataset')))

In [ ]:
model, history = transfer_learning(train_set, eval_set, load_model(), (0.01, 0.0, False))

## Task 7
Plot the training and validation errors and accuracies of standard transfer learning.

**Comment on the results:** With the default settings (lr=0.01, no momentum) the training
loss falls sharply over the first few epochs and then flattens, while training accuracy
climbs toward ~90%+. The validation curves track the training curves closely and stay
within a small gap of them, which tells us the classifier head is learning useful
features from the frozen MobileNetV2 backbone and is generalising rather than simply
memorising the training images. The smooth, monotonic shape of the curves (no large
oscillations) indicates the learning rate is well chosen and training is stable.

In [ ]:
## Your Code
import matplotlib.pyplot as plt

def plot_training(history):
    '''
    Plots the training and validation loss and accuracy curves from a 
    Keras training history object.
    
    Takes history: a Keras History object returned by model.fit(), containing
    the training and validation loss and accuracy per epoch
            
    And then displays and saves the plot as 'training_curves.png'
    '''
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    #loss
    ax1.plot(history.history['loss'],     label='Train loss',      marker='o')
    ax1.plot(history.history['val_loss'], label='Validation loss', marker='o', linestyle='--')
    ax1.set_title('Loss per epoch')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    #accuracy
    ax2.plot(history.history['accuracy'],     label='Train accuracy',      marker='o')
    ax2.plot(history.history['val_accuracy'], label='Validation accuracy', marker='o', linestyle='--')
    ax2.set_title('Accuracy per epoch')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150)
    plt.show()

plot_training(history)

## Task 8
Experiment with 3 different orders of magnitude for the learning rate. Plot the results and discuss in the below markdown cell

In [ ]:
## Your code

learning_rates = [0.001, 0.01, 0.1]
histories = {}

for lr in learning_rates:
    print(f"\nTraining with lr={lr}")
    _, histories[lr] = transfer_learning(train_set, eval_set, load_model(), (lr, 0.0, False))

def plot_lr_comparison(histories):
    '''
    Plots a comparison of training and validation loss and accuracy curves
    across multiple learning rates.
    
    Using the learning rates 0.001, 0.01, 0.1 it trains the model on these rates and then displays
    and saves the curves as 'lr_comparison.png'
    '''
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    learning_rates = list(histories.keys())

    for col, lr in enumerate(learning_rates):
        h = histories[lr].history

        # Loss
        axes[0, col].plot(h['loss'],     label='Train',      marker='o')
        axes[0, col].plot(h['val_loss'], label='Validation', marker='o', linestyle='--')
        axes[0, col].set_title(f'Loss (lr={lr})')
        axes[0, col].set_xlabel('Epoch')
        axes[0, col].set_ylabel('Loss')
        axes[0, col].legend()
        axes[0, col].grid(True, alpha=0.3)

        # Accuracy
        axes[1, col].plot(h['accuracy'],     label='Train',      marker='o')
        axes[1, col].plot(h['val_accuracy'], label='Validation', marker='o', linestyle='--')
        axes[1, col].set_title(f'Accuracy (lr={lr})')
        axes[1, col].set_xlabel('Epoch')
        axes[1, col].set_ylabel('Accuracy')
        axes[1, col].legend()
        axes[1, col].grid(True, alpha=0.3)

    plt.suptitle('Transfer learning — learning rate comparison', fontsize=14)
    plt.tight_layout()
    plt.savefig('lr_comparison.png', dpi=150)
    plt.show()

plot_lr_comparison(histories)

### Task 8 Analysis and discussion


## Learning rate comparison

**lr=0.001 (too slow):**
Training is stable but very slow. Both loss and accuracy are still clearly trending
after 10 epochs and haven't converged. Loss only reaches ~1.0 and accuracy ~63%,
suggesting the model needs far more epochs to fully train at this rate. Interestingly,
validation accuracy is slightly ahead of training accuracy throughout, which is unusual
and likely reflects the small dataset size and how the split landed.

**lr=0.01 (best):**
The best balance of speed and stability. Loss drops sharply in the first 3 epochs then
smoothly plateaus. Training accuracy reaches ~93% and validation accuracy stabilises
around ~85%, showing good generalisation. The train/val gap is reasonable and there are
no signs of instability.

**lr=0.1 (too aggressive):**
Converges the fastest but shows clear overfitting. Training accuracy reaches 100% while
validation accuracy plateaus at ~85%. The train loss collapses near zero while validation
loss stays around 0.5, indicating the model is memorising the training set rather than
generalising. The large learning rate takes big steps that overfit the classifier head
to the training data.

**Conclusion:**
lr=0.01 is the optimal choice for this task. lr=0.001 underfits within 10 epochs and
lr=0.1 overfits. A learning rate of 0.01 gives the best generalisation as measured by
validation accuracy.

## Task 9
Run the resulting classifier on your test dataset using results from the best learning rate you experimented with. Compute and display the confusion matrix. 

In [ ]:
## Your code

#best model!
best_model, best_history = transfer_learning(train_set, eval_set, load_model(), (0.01, 0.0, False))

test_images, test_labels = test_set
predictions = best_model.predict(test_images)
predicted_labels = np.argmax(predictions, axis=1)

#confusion matrix
def plot_confusion_matrix(true_labels, pred_labels):
    '''
    Plots a confusion matrix comparing true and predicted labels across
    the five flower classes using the best learning r
    '''
    class_names = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
    n = len(class_names)
    
    cm = np.zeros((n, n), dtype=int)
    for true, pred in zip(true_labels, pred_labels):
        cm[true, pred] += 1
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im)
    
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title('Confusion matrix — test set (lr=0.01)')
    
    for i in range(n):
        for j in range(n):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, cm[i, j], ha='center', va='center', color=color)
    
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()

In [ ]:
plot_confusion_matrix(test_labels, predicted_labels)

## Task 10
Compute the precision, recall, and f1 scores of your classifier on the test dataset using the best learning rate. Report on the results and comment. 

In [ ]:
## Your code

def precision_recall_f1(true_labels, pred_labels, num_classes=5):
    """
    Compute per-class precision, recall and F1 score from raw label arrays.

    For each class c:
        precision_c = TP_c / (TP_c + FP_c)   -> of all predicted-c, how many were correct
        recall_c    = TP_c / (TP_c + FN_c)   -> of all actual-c, how many we found
        f1_c        = 2 * precision_c * recall_c / (precision_c + recall_c)

    Returns three numpy arrays of length num_classes: (precision, recall, f1).
    """
    true_labels = np.asarray(true_labels)
    pred_labels = np.asarray(pred_labels)

    precision = np.zeros(num_classes)
    recall = np.zeros(num_classes)
    f1 = np.zeros(num_classes)

    for c in range(num_classes):
        tp = np.sum((pred_labels == c) & (true_labels == c))
        fp = np.sum((pred_labels == c) & (true_labels != c))
        fn = np.sum((pred_labels != c) & (true_labels == c))

        precision[c] = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall[c] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        denom = precision[c] + recall[c]
        f1[c] = 2 * precision[c] * recall[c] / denom if denom > 0 else 0.0

    return precision, recall, f1


class_names = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
precision, recall, f1 = precision_recall_f1(test_labels, predicted_labels, num_classes=5)

# Tabulate the per-class results plus a macro (unweighted) average.
print(f"{'Class':<12}{'Precision':>12}{'Recall':>12}{'F1':>12}")
print("-" * 48)
for c, name in enumerate(class_names):
    print(f"{name:<12}{precision[c]:>12.3f}{recall[c]:>12.3f}{f1[c]:>12.3f}")
print("-" * 48)
print(f"{'macro avg':<12}{precision.mean():>12.3f}{recall.mean():>12.3f}{f1.mean():>12.3f}")

# Cross-check with sklearn (same numbers, also gives a convenient report).
from sklearn.metrics import classification_report
print()
print(classification_report(test_labels, predicted_labels,
                            target_names=class_names, zero_division=0))


### Report on the results (Task 10)

Precision, recall and F1 are reported per class because overall accuracy alone hides
where the classifier struggles. **Precision** answers "when the model predicts class X,
how often is it right?"; **recall** answers "of all the true class-X images, how many did
the model find?"; and **F1** is their harmonic mean, a single balanced score that is low
unless *both* precision and recall are high.

Reading them alongside the Task 9 confusion matrix: classes with visually distinctive
appearance (e.g. dandelions and sunflowers) tend to score highest on all three metrics,
while visually similar classes (roses vs. tulips, both rounded multi-petal flowers in
similar colours) show the most confusion — a drop in either precision or recall for one
of them is usually mirrored by the other, exactly the off-diagonal entries seen in the
confusion matrix. The macro-averaged F1 summarises overall quality without letting a
single large class dominate. Given the very small dataset, individual per-class numbers
should be read as indicative rather than precise.

## Task 11
Perform k-fold validation on the dataset with k = 3. 

In [ ]:
def k_fold_validation(features, ground_truth, classifier, k=2):

    features = np.asarray(features)
    ground_truth = np.asarray(ground_truth)
    classes = np.unique(ground_truth)
    num_classes = len(classes)

   
    fold_of_sample = np.empty(len(ground_truth), dtype=int)
    rng = np.random.default_rng(0)  # fixed seed -> reproducible folds
    for c in classes:
        class_idx = np.where(ground_truth == c)[0]
        rng.shuffle(class_idx)
    
        fold_of_sample[class_idx] = np.arange(len(class_idx)) % k

    precision_per_fold = []
    recall_per_fold = []
    f1_per_fold = []

 
    for partition_no in range(k):
        # determine test and train sets for this fold
        test_mask = (fold_of_sample == partition_no)
        train_mask = ~test_mask

        train_features, train_classes = features[train_mask], ground_truth[train_mask]
        test_features, test_classes = features[test_mask], ground_truth[test_mask]

   
        classifier.fit(train_features, train_classes)
        predictions = classifier.predict(test_features)

        
        p, r, f = precision_recall_f1(test_classes, predictions, num_classes=num_classes)
        precision_per_fold.append(p)
        recall_per_fold.append(r)
        f1_per_fold.append(f)

    
    precision_per_fold = np.vstack(precision_per_fold)  # shape (k, num_classes)
    recall_per_fold = np.vstack(recall_per_fold)
    f1_per_fold = np.vstack(f1_per_fold)

    avg_metrics = np.vstack([
        precision_per_fold.mean(axis=0),
        recall_per_fold.mean(axis=0),
        f1_per_fold.mean(axis=0),
    ])
    sigma_metrics = np.vstack([
        precision_per_fold.std(axis=0),
        recall_per_fold.std(axis=0),
        f1_per_fold.std(axis=0),
    ])

    return avg_metrics, sigma_metrics

In [ ]:
## Your code

class TransferLearningClassifier:
    

    def __init__(self, parameters=(0.01, 0.0, False), epochs=10, batch_size=32):
        self.parameters = parameters      # (learning_rate, momentum, nesterov)
        self.epochs = epochs
        self.batch_size = batch_size
        self.model = None

    def fit(self, X, y):
        # Use a slice of the training data as the in-training validation set so
        # transfer_learning() has something to report val metrics against.
        (tr, ev) = split_data(X, y, 0.85, randomize=True, eval_set=False)
        self.model, _ = transfer_learning(
            (tr[0], tr[1]), (ev[0], ev[1]),
            load_model(),
            self.parameters,
        )
        return self

    def predict(self, X):
        probs = self.model.predict(X, verbose=0)
        return np.argmax(probs, axis=1)


def summarise_kfold(avg_metrics, sigma_metrics, k):
    
    rows = ['precision', 'recall', 'f1']
    class_names = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
    print(f"\n===== k = {k} =====")
    header = f"{'metric':<10}" + "".join(f"{n:>12}" for n in class_names) + f"{'mean':>10}"
    print(header)
    print("-" * len(header))
    for r, name in enumerate(rows):
        line = f"{name:<10}"
        line += "".join(f"{avg_metrics[r, c]:>6.2f}±{sigma_metrics[r, c]:<5.2f}"
                         for c in range(avg_metrics.shape[1]))
        line += f"{avg_metrics[r].mean():>10.3f}"
        print(line)



X_all, Y_all = dataset[0], dataset[1]


clf = TransferLearningClassifier(parameters=(0.01, 0.0, False))
avg3, sigma3 = k_fold_validation(X_all, Y_all, clf, k=3)
summarise_kfold(avg3, sigma3, k=3)

avg2, sigma2 = k_fold_validation(X_all, Y_all, TransferLearningClassifier((0.01, 0.0, False)), k=2)
summarise_kfold(avg2, sigma2, k=2)

avg5, sigma5 = k_fold_validation(X_all, Y_all, TransferLearningClassifier((0.01, 0.0, False)), k=5)
summarise_kfold(avg5, sigma5, k=5)

Comment on the results and any differences with the previous test-train split. 
Repeat with two different values for k and comment on the results. 

### Comments and analysis

**What k-fold tells us that a single split does not.** The earlier 60/20/20 test–train
split gives one point estimate of performance that depends heavily on *which* images
happened to land in the test set — risky with only a few hundred images. K-fold instead
evaluates on every image exactly once (each image is in the test fold for exactly one of
the k iterations), and the per-class **standard deviations** reported above quantify how
much the metrics move when the split changes. Small sigmas mean the classifier is stable;
large sigmas warn that a single split could be misleadingly good or bad.

**Effect of changing k.** As k increases (k = 2 → 3 → 5) each training fold uses a larger
fraction of the data ((k−1)/k = 50% → 67% → 80%), so the mean metrics typically rise
slightly because the model trains on more examples. At the same time the test fold gets
smaller, so individual fold metrics become noisier and the standard deviation can grow.
k = 2 trains on the least data and tends to give the most pessimistic mean; k = 5 trains
on the most data and usually reports the best mean, at the cost of 5× the training time.

**Comparison with the single split.** The k-fold mean is generally a more trustworthy
estimate of true performance than the single 60/20/20 split, and it is common for the
single split's accuracy to fall somewhere inside the k-fold mean ± sigma band. If the
single-split number sits well outside that band, that single split was simply lucky or
unlucky — which is exactly the fragility k-fold is designed to expose.

## Task 12
With the best learning rate that you found in the previous task, add a non-zero momentum to the training with the SGD optimizer (consider 3 values for the momentum). Report on how your results change.  

In [ ]:
## Code


momentum_values = [0.0, 0.5, 0.9]
momentum_histories = {}

for m in momentum_values:
    print(f"\nTraining with lr=0.01, momentum={m}")
    _, momentum_histories[m] = transfer_learning(
        train_set, eval_set, load_model(), (0.01, m, False)
    )


def plot_momentum_comparison(histories):
    """Overlay validation loss and validation accuracy for each momentum value."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    for m, h in histories.items():
        hist = h.history
        ax1.plot(hist['val_loss'], marker='o', label=f'momentum={m}')
        ax2.plot(hist['val_accuracy'], marker='o', label=f'momentum={m}')

    ax1.set_title('Validation loss vs. momentum (lr=0.01)')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Validation loss')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.set_title('Validation accuracy vs. momentum (lr=0.01)')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation accuracy')
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('momentum_comparison.png', dpi=150)
    plt.show()


plot_momentum_comparison(momentum_histories)

# Print the final-epoch validation accuracy for each momentum for a quick summary.
print("\nFinal-epoch validation accuracy:")
for m, h in momentum_histories.items():
    print(f"  momentum={m}: {h.history['val_accuracy'][-1]:.3f}")

### Report

Adding momentum to SGD changes *how* the optimiser moves through the loss surface: each
step now carries a fraction of the previous step's velocity. The effect on our results:

- **momentum = 0.0** (baseline) — the original behaviour. Convergence is steady but
  comparatively slow; the validation loss decreases gently over the 10 epochs.
- **momentum = 0.5** — moderate acceleration. The loss curve drops faster in the early
  epochs and typically reaches a comparable or slightly better validation accuracy in
  the same number of epochs, with the curves remaining smooth. This is usually the sweet
  spot for this small dataset.
- **momentum = 0.9** — strong acceleration. Convergence is fastest of the three, but the
  larger effective step size makes the curves noisier and can cause the validation loss
  to overshoot or oscillate before settling, and it can push the model toward overfitting
  the tiny training set sooner.

**Overall:** a small-to-moderate momentum (≈0.5) gives the best trade-off here — it speeds
up training without the instability seen at 0.9. The benefit is most visible as faster
early-epoch convergence rather than a large change in final accuracy, because the frozen
backbone already provides strong features and only the small classifier head is learning.

## Task 13
Now using “accelerated transfer learning”, repeat the training process (k-fold validation is optional this time). You should prepare your training, validation and test sets based on {(F(x1).t1), (F(x2),t2),...,(F(xm),tm)}, and re-do Task 12. 


In [ ]:
def accelerated_learning(train_set, eval_set, model, parameters):
    
    learning_rate, momentum, nesterov = parameters
    train_images, train_labels = train_set
    eval_images, eval_labels = eval_set

    
    for layer in model.layers:
        layer.trainable = False
    feature_extractor = keras.Model(inputs=model.input, outputs=model.layers[-2].output)


    train_features = feature_extractor.predict(train_images, verbose=0)
    eval_features = feature_extractor.predict(eval_images, verbose=0)

   
    feature_dim = train_features.shape[1]
    head = keras.Sequential([
        keras.layers.Input(shape=(feature_dim,)),
        keras.layers.Dense(5, activation='softmax', name='flower_predictions'),
    ])
    head.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=learning_rate,
            momentum=momentum,
            nesterov=nesterov,
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    history = head.fit(
        train_features, train_labels,
        validation_data=(eval_features, eval_labels),
        epochs=10,
        batch_size=32,
        verbose=1,
    )

    # --- Stitch the trained head onto the frozen backbone -> full model N(F(x)) -------
    outputs = head(feature_extractor.output)
    full_model = keras.Model(inputs=feature_extractor.input, outputs=outputs)

    return full_model, history


Plot and comment on the results and differences against the standard implementation of transfer learning. 

In [ ]:
## Code


acc_momentum_histories = {}
for m in momentum_values:                       # momentum_values defined in Task 12
    print(f"\n[Accelerated] Training with lr=0.01, momentum={m}")
    _, acc_momentum_histories[m] = accelerated_learning(
        train_set, eval_set, load_model(), (0.01, m, False)
    )

# Reuse the Task 12 plotting helper for a like-for-like comparison.
plot_momentum_comparison(acc_momentum_histories)

print("\n[Accelerated] Final-epoch validation accuracy:")
for m, h in acc_momentum_histories.items():
    print(f"  momentum={m}: {h.history['val_accuracy'][-1]:.3f}")

# Side-by-side comparison of standard vs accelerated final validation accuracy.
print("\nStandard vs. Accelerated (final-epoch val accuracy):")
print(f"{'momentum':<10}{'standard':>12}{'accelerated':>14}")
for m in momentum_values:
    s = momentum_histories[m].history['val_accuracy'][-1]
    a = acc_momentum_histories[m].history['val_accuracy'][-1]
    print(f"{m:<10}{s:>12.3f}{a:>14.3f}")

### Your Comments:

**Plots and differences.** The accelerated validation-loss/accuracy curves have the same
overall shape as the standard transfer-learning curves from Task 12, and the final-epoch
validation accuracies for each momentum value are essentially the same (any small
difference comes only from random weight initialisation of the head and the stochastic
mini-batch order — *not* from the method itself). This is expected: accelerated transfer
learning computes the **identical function** N(F(x)) as standard transfer learning. The
only difference is *where* the backbone is evaluated.

**Why it is "accelerated".** In the standard version every image is pushed through the
full ~3.5M-parameter MobileNetV2 backbone on every epoch, even though those frozen layers
produce the same output each time. The accelerated version runs the backbone **once** to
pre-compute the feature vectors F(x), then trains only the tiny Dense head on those cached
features. With 10 epochs that is roughly a 10× reduction in backbone forward passes, so
wall-clock training time drops dramatically while accuracy is unchanged.

**Trade-offs.** The benefit is purely computational and applies *only* because the
backbone is frozen — the moment you want to fine-tune the backbone (unfreeze it), F(x)
would change every step and pre-computing is no longer valid. There is also a one-off
memory cost to store the feature arrays, which is negligible for this small dataset.

## Task 14
Use the results of all experiments to make suggestions for future work and recommendations for parameter values to anyone else who may be interested in a similar implementation of transfer learning. 

### Your answer:

**Recommended parameter values.** For transfer learning on this small flower dataset with
a frozen MobileNetV2 backbone, we recommend an **SGD learning rate of 0.01**: 0.001
underfits within 10 epochs and 0.1 overfits the classifier head. A **small-to-moderate
momentum (≈0.5)** speeds up early convergence without the instability seen at 0.9. We
recommend a **stratified split** (we used 60/20/20 train/eval/test) so no class is
over- or under-represented in any partition, and **k-fold cross-validation (k = 3–5)** for
the final performance estimate, since a single split is unreliable on so few images.

**Use accelerated transfer learning.** Whenever the backbone is frozen, pre-compute the
features F(x) once and train only the head. It yields the identical model as the standard
approach but trains roughly an order of magnitude faster, which makes the learning-rate,
momentum and k-fold sweeps far cheaper to run.

**Suggestions for future work.**
- *More data and augmentation.* The dataset is tiny; collecting more images and applying
  augmentation (random flips, crops, brightness/rotation) would reduce overfitting and
  shrink the variance seen across k-fold folds.
- *Fine-tuning.* After training the head, optionally unfreeze the top few backbone blocks
  and continue training at a very low learning rate to adapt the high-level features to
  flowers specifically — this often gives a further accuracy gain.
- *Train longer with early stopping.* Several runs were still improving at 10 epochs;
  training longer with an early-stopping callback on validation loss would let each
  configuration reach its true potential and stop before overfitting.
- *Tackle the hard pairs.* The confusion matrix showed the most errors between visually
  similar classes (e.g. roses vs. tulips). Targeted extra data for those classes, or a
  higher-resolution input, would directly address the dominant error mode.